Loading Data

In [3]:
import torch

data = torch.load('data/eeg_14_70_std.pth', map_location='cpu')
print(type(data))
print(data.keys())  # see what's inside

<class 'dict'>
dict_keys(['dataset', 'labels', 'images'])


Filter to Lions vs. Pizza

In [4]:
# Labels from the perceivelab repo: lion=4, pizza=22 (verify in README)
LION_LABEL = 4
PIZZA_LABEL = 22

filtered = [x for x in data['dataset'] if x['label'] in [LION_LABEL, PIZZA_LABEL]]

# Remap to binary: 0=lion, 1=pizza
for x in filtered:
    x['binary_label'] = 0 if x['label'] == LION_LABEL else 1

print(f"Total samples: {len(filtered)}")
# Expect ~600 total (6 subjects × 50 images × 2 classes)

Total samples: 600


In [6]:
# First, check what timepoint lengths exist
lengths = [x['eeg'].shape[1] for x in filtered]
print(set(lengths))  # see all unique lengths
print(min(lengths))  # we'll crop to this

{512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 539, 540, 542, 543, 544, 545, 547, 552, 565, 576, 578, 606, 607, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511}
491


In [7]:

MIN_LEN = min(lengths)  # = 491
OCCIPITAL_CHANNELS = list(range(75, 96))

X = np.array([x['eeg'][OCCIPITAL_CHANNELS, :MIN_LEN].detach().numpy() for x in filtered])
y = np.array([x['binary_label'] for x in filtered])
print(X.shape, y.shape)

(600, 21, 491) (600,)


Select Occipital Channels

In [9]:
# Check if the dataset has channel location info
print(data.keys())
print(data['stddev'] if 'stddev' in data else "no stddev")

# Also check what keys each trial has
print(filtered[0].keys())

dict_keys(['dataset', 'labels', 'images'])
no stddev
dict_keys(['eeg', 'image', 'label', 'subject', 'binary_label'])


In [10]:
import collections
labels = [x['label'] for x in filtered]
print(collections.Counter(labels))

binary = [x['binary_label'] for x in filtered]
print(collections.Counter(binary))

Counter({4: 300, 22: 300})
Counter({0: 300, 1: 300})


In [11]:
print(data['labels'])

['n02389026', 'n03888257', 'n03584829', 'n02607072', 'n03297495', 'n03063599', 'n03792782', 'n04086273', 'n02510455', 'n11939491', 'n02951358', 'n02281787', 'n02106662', 'n04120489', 'n03590841', 'n02992529', 'n03445777', 'n03180011', 'n02906734', 'n07873807', 'n03773504', 'n02492035', 'n03982430', 'n03709823', 'n03100240', 'n03376595', 'n03877472', 'n03775071', 'n03272010', 'n04069434', 'n03452741', 'n03792972', 'n07753592', 'n13054560', 'n03197337', 'n02504458', 'n02690373', 'n03272562', 'n04044716', 'n02124075']


In [12]:
# Map the two labels you're using
synset_map = {
    4: data['labels'][4],   # your "lion"
    22: data['labels'][22]  # your "pizza"
}
print(synset_map)

# Decode them
known = {
    'n03297495': 'espresso maker',
    'n03982430': 'pool table / billiards',
    # we need to look these up
}
print("Label 4 synset:", data['labels'][4])
print("Label 22 synset:", data['labels'][22])

{4: 'n03297495', 22: 'n03982430'}
Label 4 synset: n03297495
Label 22 synset: n03982430


In [13]:
import numpy as np

OCCIPITAL_CHANNELS = list(range(75, 96))  # adjust after verifying

X = np.array([x['eeg'][OCCIPITAL_CHANNELS, :] for x in filtered])  # shape: (N, channels, timepoints)
y = np.array([x['binary_label'] for x in filtered])
print(X.shape, y.shape)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (600, 21) + inhomogeneous part.

In [14]:
# Print all 40 labels with their index so we can find lion and pizza
for i, label in enumerate(data['labels']):
    print(i, label)

0 n02389026
1 n03888257
2 n03584829
3 n02607072
4 n03297495
5 n03063599
6 n03792782
7 n04086273
8 n02510455
9 n11939491
10 n02951358
11 n02281787
12 n02106662
13 n04120489
14 n03590841
15 n02992529
16 n03445777
17 n03180011
18 n02906734
19 n07873807
20 n03773504
21 n02492035
22 n03982430
23 n03709823
24 n03100240
25 n03376595
26 n03877472
27 n03775071
28 n03272010
29 n04069434
30 n03452741
31 n03792972
32 n07753592
33 n13054560
34 n03197337
35 n02504458
36 n02690373
37 n03272562
38 n04044716
39 n02124075


Build the CNN

In [ ]:
import torch.nn as nn

class EEG_CNN(nn.Module):
    def __init__(self, n_channels, n_timepoints):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(1, 25), padding=(0, 12)),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.Conv2d(16, 32, kernel_size=(n_channels, 1)),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(0.5)
        )
        self.fc = nn.Linear(32 * (n_timepoints // 4), 1)

    def forward(self, x):
        x = x.unsqueeze(1)       # add channel dim
        x = self.conv(x)
        x = x.flatten(1)
        return self.fc(x)

Train & Evaluate 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors, train loop, track accuracy per epoch
# Plot: training curve + confusion matrix with matplotlib